# Trustpilot Sentiment Analyzer

End-to-end pipeline: scrape reviews → tag sentiment + topics → synthesize PM insights → generate report.

**Before running:** copy `.env.example` to `.env` and add your `ANTHROPIC_API_KEY`.

In [ ]:
# Install dependencies (uncomment if running in Google Colab)
# !pip install anthropic beautifulsoup4 matplotlib pandas python-dotenv requests

## 1. Configuration

In [ ]:
from scraper import TrustpilotScraper
from analyzer import ReviewAnalyzer
from report import ReportGenerator

# --- Configure your run here ---

# Trustpilot URL for the company you want to analyze.
# Tip: append ?stars=1 to focus on 1-star reviews, ?stars=5 for 5-star, etc.
TRUSTPILOT_URL = "https://www.trustpilot.com/review/webuyanycarusa.com"

# Set a page cap to limit API costs during testing. None = scrape all pages.
MAX_PAGES = 5

print(f"Target URL : {TRUSTPILOT_URL}")
print(f"Max pages  : {MAX_PAGES if MAX_PAGES else 'all'}")

## 2. Scrape Reviews

In [ ]:
scraper = TrustpilotScraper(delay=1.5)
df_raw = scraper.scrape(TRUSTPILOT_URL, max_pages=MAX_PAGES)
df_raw.head()

In [ ]:
print(f"Reviews scraped : {len(df_raw)}")
print(f"Rating breakdown:\n{df_raw['rating'].value_counts().sort_index()}")

## 3. Analyze — Sentiment Tagging + Theme Extraction

Two-model approach:
- **Claude Haiku** tags each review with sentiment + topic (fast, cheap)
- **Claude Sonnet** synthesizes themes and generates PM recommendations (better reasoning)

In [ ]:
analyzer = ReviewAnalyzer()
df_analyzed, insights = analyzer.analyze(df_raw)

In [ ]:
# Preview tagged reviews
df_analyzed[["reviewer", "rating", "sentiment", "topic", "key_point"]].head(10)

## 4. PM Insights

In [ ]:
import json
print(json.dumps(insights, indent=2))

## 5. Generate Report + Charts

In [ ]:
reporter = ReportGenerator(output_dir="output")
report_md = reporter.generate_report(df_analyzed, insights)

# Display inline
from IPython.display import Markdown
Markdown(report_md)

In [ ]:
# Show charts inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

for chart in ["sentiment_distribution.png", "topic_breakdown.png", "rating_trend.png"]:
    path = os.path.join("output", chart)
    if os.path.exists(path):
        img = mpimg.imread(path)
        plt.figure(figsize=(10, 5))
        plt.imshow(img)
        plt.axis("off")
        plt.tight_layout()
        plt.show()

## 6. Export Data

In [ ]:
company = df_analyzed["company"].iloc[0].replace(" ", "_").lower()
csv_path = f"output/{company}_reviews_analyzed.csv"
df_analyzed.to_csv(csv_path, index=False)
print(f"Data exported to {csv_path}")